# Kill Disagg Sessions

Identify and kill stale prefiller / decoder / proxy processes left over from interrupted runs.

In [15]:
import subprocess, os, signal, time

PORTS = [7100, 7200, 7300, 7400, 7500, 9101]

PATTERNS = [
    'vllm serve',
    'vllm.entrypoints',
    'disagg_proxy_server',
    'unit_disagg_profile',
    'launch_disagg_profile',
]

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

def _gpu_pids():
    """PIDs currently using GPU memory (nvidia-smi)."""
    out = sh("nvidia-smi --query-compute-apps=pid --format=csv,noheader 2>/dev/null")
    return {int(p) for p in out.split() if p.strip().isdigit()}

def _tree_pids():
    """All PIDs in the process tree of any matching pattern process."""
    try:
        import psutil
    except ImportError:
        return set()
    roots = set()
    for proc in psutil.process_iter(['pid', 'cmdline']):
        try:
            cmd = ' '.join(proc.info['cmdline'] or [])
            if any(p in cmd for p in PATTERNS):
                roots.add(proc.pid)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    all_pids = set(roots)
    for pid in roots:
        try:
            for child in psutil.Process(pid).children(recursive=True):
                all_pids.add(child.pid)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return all_pids

def _port_pids():
    """PIDs holding any of the disagg ports open."""
    pids = set()
    for port in PORTS:
        for p in sh(f"lsof -ti :{port} 2>/dev/null").split():
            if p.isdigit():
                pids.add(int(p))
    return pids

def _candidate_pids():
    """Union of pattern-matched trees, port holders, and GPU workers."""
    pids = set()
    # pattern-matched processes (pgrep)
    for pat in PATTERNS:
        for p in sh(f"pgrep -f '{pat}' 2>/dev/null").split():
            if p.isdigit():
                pids.add(int(p))
    pids |= _tree_pids()
    pids |= _port_pids()
    # all processes holding GPU memory — includes orphaned EngineCore workers
    pids |= _gpu_pids()
    pids.discard(os.getpid())
    return pids

def show():
    """Print current disagg process state without killing anything."""
    pat = '|'.join(PATTERNS)
    procs = sh(f"ps aux | grep -E '{pat}' | grep -v grep")
    print("=== Disagg processes ===")
    print(procs or "(none)")
    print()
    print("=== GPU compute apps ===")
    print(sh("nvidia-smi --query-compute-apps=pid,used_memory,name --format=csv,noheader 2>/dev/null") or "(none)")
    print()
    port_str = ' '.join(f'-ti :{p}' for p in PORTS)
    print(f"=== Port holders ({' '.join(str(p) for p in PORTS)}) ===")
    holders = sh(f"lsof {port_str} 2>/dev/null | sort -u")
    print(holders or "(none)")

def kill_all(dry_run=False):
    """Kill all prefiller / decoder / proxy processes and their spawn workers."""
    pids = _candidate_pids()
    if not pids:
        print("Nothing to kill.")
        return
    label = "[DRY RUN] Would kill" if dry_run else "Killing"
    print(f"{label} PIDs: {sorted(pids)}")
    if dry_run:
        return
    # SIGTERM first, then SIGKILL after grace period
    for pid in sorted(pids):
        try: os.kill(pid, signal.SIGTERM)
        except ProcessLookupError: pass
    time.sleep(3)
    for pid in sorted(pids):
        try: os.kill(pid, signal.SIGKILL)
        except ProcessLookupError: pass
    time.sleep(1)
    print()
    remaining = sh("nvidia-smi --query-compute-apps=pid,used_memory,name --format=csv,noheader 2>/dev/null")
    print("=== GPU after kill ===")
    print(remaining or "GPUs clear")

show()


=== Disagg processes ===
eicchen   140249  0.0  0.0   7640  4176 pts/28   S+   16:32   0:00 bash conserve_project/conserve/profiling/launch_disagg_profile.sh
eicchen   140275 11.7  0.1 11955284 963276 pts/28 Sl+ 16:32   0:12 /data/projects/eicchen/miniconda3/envs/conserve-Qwen3-0.6B/bin/python3.10 /data/projects/eicchen/miniconda3/envs/conserve-Qwen3-0.6B/bin/vllm serve Qwen/Qwen3-0.6B --port 7200 --download-dir /data/projects/eicchen/conserve_project/conserve/models_download --rope-scaling {"rope_type":"dynamic","factor":2.0} --max-num-batched-tokens 2944 --max-num-seqs 1024 --disable-log-requests --enforce-eager --no-enable-prefix-caching --kv-transfer-config {"kv_connector":"LMCacheConnectorV1","kv_role":"kv_consumer","kv_connector_extra_config": {"discard_partial_chunks": false, "lmcache_rpc_port": "consumer1", "skip_last_n_tokens": 1}} --engine-log-file /data/projects/eicchen/conserve_project/conserve/profiling/gpu_profiling/A40/Qwen3-0.6B/pd_disagg_300W/128/decoder_vllm_engine_lo

In [22]:
kill_all()


Killing PIDs: [281908, 281910, 281912, 281914, 281916]

=== GPU after kill ===
GPUs clear


In [21]:
print(sh("nvidia-smi"))


Thu Jun 25 16:50:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.42.06              Driver Version: 555.42.06      CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:01:00.0 Off |                    0 |
|  0%   29C    P8             21W /  300W |       4MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----